# Residuales, incertidumbre e identificabilidad

**Nivel 2 · Prerrequisitos: Problemas inversos: evidencia y falta de unicidad; lectura: 25 minutes.**

## Objetivos

Separe residuales con signo, error normalizado, supuestos de incertidumbre y falta de unicidad del modelo. El resultado versionado del producto contiene métricas de entrenamiento/holdout y varias alternativas de profundidad matemática aceptadas; inspeccione esos resultados sin reconstruirlos.

## Preparación

Use el entorno de notebooks Python 3.12 revisado del repositorio. Este notebook no instala dependencias. El artefacto es un caso sintético de prisma generado por el producto. Los errores son resúmenes almacenados; esta lección no inventa arreglos de observaciones.

La celda siguiente lee las métricas guardadas de train/holdout y el registro de no unicidad.

In [ ]:
import json
from pathlib import Path

# Leer métricas calculadas del producto y modelos alternativos; no sintetizar observaciones.
root = Path.cwd().parents[2]
caso = root / 'data/derived/m01-scientific-course/prism-case-0'
resultado = json.loads((caso / 'result.json').read_text(encoding='utf-8'))
for particion in ('train', 'holdout'):
    print(particion, resultado['evaluation'][particion])
print('contrato de incertidumbre:', resultado['uncertainty'])
print('alternativas solo de entrenamiento:')
for modelo in resultado['nonuniqueness']['training_only_alternatives']:
    print({k: modelo[k] for k in ('depth_m', 'damping', 'training_rmse_mgal', 'status')})

## Datos

El artefacto es un caso sintético de prisma generado por el producto. Los errores son resúmenes almacenados; esta lección no inventa arreglos de observaciones.

**Archivos:** `data/derived/m01-scientific-course/prism-case-0/result.json`, `receipt.json`.

## Método

Para el residual r=d−F(m), RMSE conserva las unidades de los datos; el RMSE normalizado divide por la escala de ruido declarada. Un valor normalizado cercano a uno no basta si hay observaciones sin soporte o correlacionadas. La incertidumbre reportada es propagación lineal condicional con pesos diagonales de estaciones independientes; excluye geometría, selección de parámetros, sesgo e incertidumbre geológica. SimPEG describe el contexto de modelado directo y estimación de parámetros ([Cockett et al., 2015](https://doi.org/10.1016/j.cageo.2015.09.015)); las métricas numéricas aquí provienen del artefacto del producto.

Para residual r=d−F(m), errores gaussianos diagonales dan φ_d=½∑ᵢ(rᵢ/σᵢ)². El residual estandarizado rᵢ/σᵢ es adimensional; el RMSE conserva la unidad de los datos. Con errores correlacionados se usa rᵀC_d⁻¹r, no una suma diagonal. El caso del producto supone explícitamente errores independientes por estación y excluye error geométrico, incertidumbre de selección, sesgo e incertidumbre geológica; por tanto, su σ propagada es condicional, no un intervalo geológico posterior. Esta distinción es central en la estimación de parámetros inversos [Cockett et al. (2015)](https://doi.org/10.1016/j.cageo.2015.09.015).

El holdout espacial importa porque observaciones cercanas de campos potenciales comparten respuestas suaves. Retener puntos aislados al azar puede dejar casi duplicados en entrenamiento y exagerar la capacidad de transferencia. Este caso guardado tiene 149 filas train, 43 holdout externo y cuatro enmascaradas; solo 30 predicciones holdout tienen soporte. El RMSE normalizado es cerca de 0.735 en train y 1.289 en holdout con soporte; 13 filas holdout no tienen soporte. Profundidades matemáticas alternativas de 160, 360 y 700 m tienen distinto amortiguamiento y ajuste. “Aprobado” indica que pasó una puerta numérica acotada; la profundidad es parámetro de base, no profundidad física de fuente.

## Resultados

El RMSE normalizado del holdout supera al de entrenamiento y hay 13 filas de holdout sin soporte. Varias alternativas de profundidad solo de entrenamiento aparecen como aprobadas con distinto amortiguamiento y ajuste. Esto evidencia ambigüedad condicional; no autoriza elegir una geología preferida. Interprete la salida junto con el soporte de datos y las exclusiones de incertidumbre.

Estos paneles comparan candidatos guardados de CV bloqueada con la evaluación externa del producto. Los candidatos fallidos no tienen puntuación y se omiten en vez de asignarles una penalización inventada. La figura diagnostica el caso sintético creado.


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

# Graficar candidatos CV y métricas train/holdout persistidos, sin regenerar el ajuste.
ruta = Path.cwd().parents[2] / 'data/derived/m01-scientific-course/prism-case-0/result.json'
a = json.loads(ruta.read_text(encoding='utf-8'))
fig, (ax,bx) = plt.subplots(1,2,figsize=(11,4.8),constrained_layout=True)
for amort in sorted({c['damping'] for c in a['selection']['candidates']}):
 cs=[c for c in a['selection']['candidates'] if c['damping']==amort and c.get('cv_normalized_rmse') is not None]
 ax.plot([c['depth_m'] for c in cs],[c['cv_normalized_rmse'] for c in cs],marker='o',label=f'amortiguamiento={amort:g}')
ax.set(xlabel='Profundidad de base armónica [m]',ylabel='RMSE normalizado CV bloqueada [1]',title='Puntuaciones guardadas'); ax.legend()
etiq=['Train','Holdout con soporte']; vals=[a['evaluation'][k]['normalized_rmse'] for k in ('train','holdout')]
bx.bar(etiq,vals,color=['#4878a8','#d65f5f']); bx.set(ylabel='RMSE normalizado [1]',title='Evaluación externa')
for i,k in enumerate(('train','holdout')):
 m=a['evaluation'][k]; bx.text(i,vals[i],f"n={m['covered_count']} con soporte; {m['unsupported_count']} sin soporte",ha='center',va='bottom',fontsize=8)
print('condición del diseño ponderado:',a['condition']['weighted_design_condition'])
print('exclusiones de incertidumbre:',a['uncertainty']['excludes'])
plt.show()


## Resumen

Un residual pequeño describe predicciones bajo un experimento y esquema de ponderación. No prueba un modelo único ni incertidumbre geológica calibrada.


## Referencias

- https://doi.org/10.1016/j.cageo.2015.09.015